# Configuração do Ambiente

In [0]:
%pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:

########################################################
#### LIBS
########################################################
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    BooleanType, DoubleType, TimestampType, LongType
)
from pyspark.sql.window import Window
import pyspark
#from pyspark.sql.functions import col
import pandas as pd
import numpy as np
import requests
import json
import os
import hashlib
import json as _json
from datetime import datetime, timezone

# PIB IBGE

In [0]:
###pib dos municípios, segundo o IBGE
pib_ibge = pd.read_csv("/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/batch/br_ibge_pib_municipio.csv")
pib_ibge = pib_ibge[["id_municipio","ano", "pib"]]# seleção no pandas
pib_ibge = pib_ibge[pib_ibge["ano"] == 2023]


In [0]:
###
pib_ibge.shape

In [0]:
##conversão em DataFrame Spark
pib_ibge = spark.createDataFrame(pib_ibge)
pib_ibge.show(5)

## Tratamento dos dados

In [0]:
pib_ibge = pib_ibge.withColumnRenamed("id_municipio", "CD_MUN")

# Censo IBGE 2022

In [0]:
# evita o erro de tipos misturados
ibge = pd.read_excel("/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/batch/ibge_agregados_municipios_basico_BR.xlsx", dtype=str) 
ibge = ibge[["CD_MUN", "NM_MUN", "CD_REGIAO", "NM_REGIAO", "CD_UF", "NM_UF", "AREA_KM2", "v0001"]]# seleção no pandas
ibge = ibge.where(ibge.notna(), None) #NaN -> None

##conversão em DataFrame Spark
ibge = spark.createDataFrame(ibge)

## Tratamento dos dados

In [0]:
##
ibge = ibge.withColumnRenamed("v0001", "POPULACAO")

#Conversão do tipo da coluna
ibge = (
    ibge
    .withColumn("AREA_KM2", F.col("AREA_KM2").cast("double"))  
    .withColumn("POPULACAO",    F.col("POPULACAO").cast("long"))     
)

ibge.show(5, truncate=False)

# Índice de Desenvolvimento da Educação Básica (IDEB)

## Conversão do excel para um DataFrame Spark

In [0]:
# evita o erro de tipos misturados
ideb = pd.read_excel("/Volumes/workspace/default/inep_avaliacao_alfabetizacao/bronze/batch/divulgacao_anos_iniciais_municipios_2025.xlsx", dtype=str) 
ideb = ideb.where(ideb.notna(), None) #NaN -> None

##conversão em DataFrame Spark
ideb = spark.createDataFrame(ideb)

##
ideb = ideb.dropna(how="all")
ideb = ideb.replace(["ND", "-"], None)

## Tratamento dos dados

In [0]:
##
ideb = (
    ideb
    .withColumnRenamed("Sigla da UF", "sigla_uf")
    .withColumnRenamed("Código do Município", "CD_MUN")
    .withColumnRenamed("Nome do Município", "NM_MUN2")
    .withColumnRenamed("Rede", "rede")
    .withColumnRenamed("Matemática", "matematica_saeb")
    .withColumnRenamed("Língua Portuguesa", "portugues_saeb")
    .withColumnRenamed("IDEB\n2023", "ideb_2023")
)

##
ideb = (
    ideb
    .withColumn("matematica_saeb", F.col("matematica_saeb").try_cast("double"))    
    .withColumn("portugues_saeb",    F.col("portugues_saeb").try_cast("double"))  
    .withColumn("ideb_2023",    F.col("ideb_2023").try_cast("double"))       
)

ideb.show(5)

# Join das duas bases

In [0]:
resultado = ibge.join(ideb, on = "CD_MUN", how = "left")
resultado = resultado.join(pib_ibge, on = "CD_MUN", how = "left")
resultado = resultado.select("CD_MUN", "NM_MUN", "NM_REGIAO", "sigla_uf", "AREA_KM2", "POPULACAO", 
                             "rede", "matematica_saeb", "portugues_saeb", "ideb_2023", "pib")
resultado.show(5)

# Exportação dos dados

In [0]:

#
file_path =  "/Volumes/workspace/default/inep_avaliacao_alfabetizacao/silver/ibge_ideb"
#
resultado.write.mode("overwrite").parquet(file_path)